In [ ]:
from data_loader import load_dataset
thetas, ps2d, xhi = load_dataset(results_dir="low_generate_data/results")

In [1]:
import torch
from train      import train
from false_data import make_dataset
from evaluate   import run_inference, print_metrics, plot_pred_vs_true, \
                       plot_residuals, plot_ps2d_maps

N_TRAIN, N_VAL, N_TEST = 20_000, 400, 1_000

def main():
    # ── Data ──────────────────────────────────────────────────────────────────
    print("Generating datasets...")
    train_thetas, train_ps2d, train_xhi, _ = make_dataset(N_TRAIN, seed=0)
    val_thetas,   val_ps2d,   val_xhi,   _ = make_dataset(N_VAL,   seed=1)
    test_thetas,  test_ps2d,  test_xhi,  _ = make_dataset(N_TEST,  seed=42)

    # ── Training ──────────────────────────────────────────────────────────────
    print("\nTraining emulator...")
    model = train(
        train_thetas, train_ps2d, train_xhi,
        epochs=300, batch_size=256, lr=1e-3,
        w_ps=1.0, w_xhi=1.0,
        checkpoint_dir="emulator/checkpoints",
    )

    # ── Evaluation ────────────────────────────────────────────────────────────
    print("\nEvaluating on test set...")
    ps2d_pred, xhi_pred = run_inference(model, test_thetas)
    ps2d_true = test_ps2d.numpy()
    xhi_true  = test_xhi.numpy()

    print_metrics(ps2d_pred, ps2d_true, xhi_pred, xhi_true)
    plot_pred_vs_true(ps2d_pred, ps2d_true, xhi_pred, xhi_true)
    plot_residuals   (ps2d_pred, ps2d_true, xhi_pred, xhi_true)
    plot_ps2d_maps   (ps2d_pred, ps2d_true, sample_idx=0)

    print("\nDone.")

if __name__ == "__main__":
    main()

Generating datasets...

Training emulator...
epoch   50/300  loss=0.046543
epoch  100/300  loss=0.027933
epoch  150/300  loss=0.002745


: 